# Missing Data Analysis

This notebook analyzes missing values in Test.csv where missing is encoded as -9999.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

%matplotlib inline

In [ ]:
df = pd.read_csv('../data/Test.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# Identify feature columns (excluding ID)
feature_cols = [c for c in df.columns if c != 'ID']
print('Number of feature columns:', len(feature_cols))

# Assume 12 months, each with same number of features
n_months = 12
n_features_per_month = len(feature_cols) // n_months
print('Features per month:', n_features_per_month)

# Reshape to (n_rows, n_months, n_features_per_month)
data = df[feature_cols].values.reshape((df.shape[0], n_months, n_features_per_month))

missing_val = -9999

# Create masks for the three states of interest
missing_mask = (data == missing_val)  # True where value is -9999

# State 1: All 12 features present
all_present = ~np.any(missing_mask, axis=2)

# State 2: Only VV and VH features present (others missing)
vv_vh_present_others_missing = np.zeros((data.shape[0], data.shape[1]), dtype=bool)
for month in range(12):
    # VH and VV are at indices 0 and 1 within each month's 12-feature block
    vh_present = ~missing_mask[:, month, 0]   # VH present
    vv_present = ~missing_mask[:, month, 1]   # VV present
    others_missing = np.all(missing_mask[:, month, 2:], axis=1)  # Others missing
    vv_vh_present_others_missing[:, month] = vh_present & vv_present & others_missing

# State 3: None of the features present
none_present = np.all(missing_mask, axis=2)

# For backward compatibility and additional insights:
# Months with ANY data (either all present or only VV/VH present)
months_with_any_data = (all_present | vv_vh_present_others_missing).sum(axis=1)
# Months with COMPLETE data (all 12 present)
months_with_complete_data = all_present.sum(axis=1)
# Months with ONLY VV/VH present
months_with_only_vv_vh = vv_vh_present_others_missing.sum(axis=1)
# Months with NO data
months_with_no_data = none_present.sum(axis=1)

print('First 5 rows months_with_any_data:', months_with_any_data[:5])
print('First 5 rows months_with_complete_data:', months_with_complete_data[:5])
print('First 5 rows months_with_only_vv_vh:', months_with_only_vv_vh[:5])
print('First 5 rows months_with_no_data:', months_with_no_data[:5])

print('\nSummary for months with ANY data:')
s_any = pd.Series(months_with_any_data)
print(s_any.describe())
print('Mean months with any data per row:', s_any.mean())

print('\nSummary for months with COMPLETE data:')
s_complete = pd.Series(months_with_complete_data)
print(s_complete.describe())
print('Mean months with complete data per row:', s_complete.mean())

print('\nSummary for months with ONLY VV/VH:')
s_vv_vh = pd.Series(months_with_only_vv_vh)
print(s_vv_vh.describe())
print('Mean months with only VV/VH per row:', s_vv_vh.mean())

print('\nSummary for months with NO data:')
s_none = pd.Series(months_with_no_data)
print(s_none.describe())
print('Mean months with no data per row:', s_none.mean())

In [ ]:
# Per-month availability breakdown
# Calculate percentages for each state
all_present_pct = all_present.mean(axis=0) * 100
vv_vh_only_pct = vv_vh_present_others_missing.mean(axis=0) * 100
none_present_pct = none_present.mean(axis=0) * 100

print('Percentage of rows by state per month:')
for i in range(12):
    month = i + 1
    print(f'Month {month:02d}:')
    print(f'  All 12 features present: {all_present_pct[i]:.2f}%')
    print(f'  Only VV/VH present:      {vv_vh_only_pct[i]:.2f}%')
    print(f'  No features present:     {none_present_pct[i]:.2f}%')
    print(f'  Any data present:        {all_present_pct[i] + vv_vh_only_pct[i]:.2f}%')
    print()

# Plot stacked bar chart showing the three states
plt.figure(figsize=(12, 6))
months = range(1, 13)
p1 = plt.bar(months, all_present_pct, label='All 12 features present', color='green', alpha=0.8)
p2 = plt.bar(months, vv_vh_only_pct, bottom=all_present_pct, label='Only VV/VH present', color='orange', alpha=0.8)
p3 = plt.bar(months, none_present_pct, bottom=all_present_pct + vv_vh_only_pct, label='No features present', color='red', alpha=0.8)

plt.xlabel('Month')
plt.ylabel('Percentage of rows')
plt.title('Data Availability by Month - Three State Breakdown')
plt.xticks(months)
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()

# Also show the traditional 'any data' view for comparison
any_data_pct = (all_present_pct + vv_vh_only_pct)
plt.figure(figsize=(10, 5))
plt.bar(months, any_data_pct, color='skyblue', alpha=0.8)
plt.xlabel('Month')
plt.ylabel('Percentage of rows with any data')
plt.title('Data Availability by Month (Any Data Present)')
plt.xticks(months)
plt.ylim(0, 100)
plt.show()

In [ ]:
# Number of months by state per row
print('Months count statistics by state:')
print()
print('ANY data (all present OR only VV/VH):')
print(pd.Series(months_with_any_data).describe())
print()
print('COMPLETE data (all 12 present):')
print(pd.Series(months_with_complete_data).describe())
print()
print('ONLY VV/VH present:')
print(pd.Series(months_with_only_vv_vh).describe())
print()
print('NO data present:')
print(pd.Series(months_with_no_data).describe())
print()

print('Total counts per month for months with ANY data:')
for month in range(1, 13):
    count = (np.array(months_with_any_data) == month).sum()
    print(f'Month {month}: {count}')

# Distribution plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Any data
axes[0, 0].hist(months_with_any_data, bins=range(0, 14), alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].set_xlabel('Number of months')
axes[0, 0].set_ylabel('Number of rows')
axes[0, 0].set_title('Distribution of Months with ANY Data')
axes[0, 0].grid(alpha=0.3)

# Complete data
axes[0, 1].hist(months_with_complete_data, bins=range(0, 14), alpha=0.7, color='green', edgecolor='black')
axes[0, 1].set_xlabel('Number of months')
axes[0, 1].set_ylabel('Number of rows')
axes[0, 1].set_title('Distribution of Months with COMPLETE Data')
axes[0, 1].grid(alpha=0.3)

# Only VV/VH
axes[1, 0].hist(months_with_only_vv_vh, bins=range(0, 14), alpha=0.7, color='orange', edgecolor='black')
axes[1, 0].set_xlabel('Number of months')
axes[1, 0].set_ylabel('Number of rows')
axes[1, 0].set_title('Distribution of Months with ONLY VV/VH Data')
axes[1, 0].grid(alpha=0.3)

# No data
axes[1, 1].hist(months_with_no_data, bins=range(0, 14), alpha=0.7, color='red', edgecolor='black')
axes[1, 1].set_xlabel('Number of months')
axes[1, 1].set_ylabel('Number of rows')
axes[1, 1].set_title('Distribution of Months with NO Data')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Let's print the rows with 10 months of any data (i.e., only 2 months with any data)
rows_with_10_missing_any = np.where(months_with_any_data == 2)[0]  # 12-10 = 2 months with any data
print(f'Rows with 10 months missing any data (only 2 months with any data): {len(rows_with_10_missing_any)}')
if len(rows_with_10_missing_any) > 0:
    print('Row indices:', rows_with_10_missing_any[:10], '...' if len(rows_with_10_missing_any) > 10 else '')

# Let's also look at rows with 10 months of complete data
rows_with_10_missing_complete = np.where(months_with_complete_data == 2)[0]
print(f'\nRows with 10 months missing complete data (only 2 months complete): {len(rows_with_10_missing_complete)}')
if len(rows_with_10_missing_complete) > 0:
    print('Row indices:', rows_with_10_missing_complete[:10], '...' if len(rows_with_10_missing_complete) > 10 else '')

# And rows with 10 months of only VV/VH data
rows_with_10_missing_vv_vh = np.where(months_with_only_vv_vh == 2)[0]
print(f'\nRows with 10 months missing only VV/VH data (only 2 months with VV/VH): {len(rows_with_10_missing_vv_vh)}')
if len(rows_with_10_missing_vv_vh) > 0:
    print('Row indices:', rows_with_10_missing_vv_vh[:10], '...' if len(rows_with_10_missing_vv_vh) > 10 else '')

In [ ]:
# Longest consecutive months by state per row
def max_consecutive_true(arr):
    max_len = cur = 0
    for v in arr:
        if v:
            cur += 1
            max_len = max(max_len, cur)
        else:
            cur = 0
    return max_len

# Calculate for each state
max_consec_any = np.apply_along_axis(max_consecutive_true, 1, (all_present | vv_vh_present_others_missing))
max_consec_complete = np.apply_along_axis(max_consecutive_true, 1, all_present)
max_consec_vv_vh = np.apply_along_axis(max_consecutive_true, 1, vv_vh_present_others_missing)
max_consec_none = np.apply_along_axis(max_consecutive_true, 1, none_present)

print('Maximum consecutive months statistics:')
print()
print('ANY data (all present OR only VV/VH):')
print(pd.Series(max_consec_any).describe())
print()
print('COMPLETE data (all 12 present):')
print(pd.Series(max_consec_complete).describe())
print()
print('ONLY VV/VH present:')
print(pd.Series(max_consec_vv_vh).describe())
print()
print('NO data present:')
print(pd.Series(max_consec_none).describe())
print()

# Plot distribution of max consecutive months
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(max_consec_any, bins=range(0, 14), alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].set_xlabel('Max consecutive months')
axes[0, 0].set_ylabel('Number of rows')
axes[0, 0].set_title('Distribution of Max Consecutive Months with ANY Data')
axes[0, 0].grid(alpha=0.3)

axes[0, 1].hist(max_consec_complete, bins=range(0, 14), alpha=0.7, color='green', edgecolor='black')
axes[0, 1].set_xlabel('Max consecutive months')
axes[0, 1].set_ylabel('Number of rows')
axes[0, 1].set_title('Distribution of Max Consecutive Months with COMPLETE Data')
axes[0, 1].grid(alpha=0.3)

axes[1, 0].hist(max_consec_vv_vh, bins=range(0, 14), alpha=0.7, color='orange', edgecolor='black')
axes[1, 0].set_xlabel('Max consecutive months')
axes[1, 0].set_ylabel('Number of rows')
axes[1, 0].set_title('Distribution of Max Consecutive Months with ONLY VV/VH Data')
axes[1, 0].grid(alpha=0.3)

axes[1, 1].hist(max_consec_none, bins=range(0, 14), alpha=0.7, color='red', edgecolor='black')
axes[1, 1].set_xlabel('Max consecutive months')
axes[1, 1].set_ylabel('Number of rows')
axes[1, 1].set_title('Distribution of Max Consecutive Months with NO Data')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Check if missing pattern is monotonic (missing only at start or end) for ANY data missing
def is_monotonic_missing(arr):
    # Find first and last missing (False values)
    idx = np.where(~arr)[0]  # Where condition is False
    if len(idx) == 0:
        return True  # no missing
    return (idx[-1] - idx[0] + 1) == len(idx)  # contiguous block

monotonic_any = np.apply_along_axis(is_monotonic_missing, 1, (all_present | vv_vh_present_others_missing))
monotonic_complete = np.apply_along_axis(is_monotonic_missing, 1, all_present)
monotonic_vv_vh = np.apply_along_axis(is_monotonic_missing, 1, vv_vh_present_others_missing)
monotonic_none = np.apply_along_axis(is_monotonic_missing, 1, none_present)

print('Proportion of rows with missing months forming a single contiguous block:')
print(f'ANY data missing:  {monotonic_any.mean():.3f}')
print(f'COMPLETE data missing: {monotonic_complete.mean():.3f}')
print(f'ONLY VV/VH missing: {monotonic_vv_vh.mean():.3f}')
print(f'NO data missing:   {monotonic_none.mean():.3f}')
print()

# Additional: count rows with missing at start only, end only, middle gaps for ANY data
any_missing = ~(all_present | vv_vh_present_others_missing)
start_only = np.zeros(any_missing.shape[0], dtype=bool)
end_only = np.zeros(any_missing.shape[0], dtype=bool)
middle_gap = np.zeros(any_missing.shape[0], dtype=bool)

for i, row in enumerate(any_missing):
    if not np.any(row):
        continue  # no missing
    first = np.where(row)[0][0]
    last = np.where(row)[0][-1]
    if first == 0 and last == np.sum(row)-1:
        # missing from start to some point
        # check if missing are at start (i.e., missing indices are 0..k)
        if np.all(np.arange(0, np.sum(row)) == np.where(row)[0]):
            start_only[i] = True
    if last == 11 and first == np.sum(row)-1 + (11 - np.sum(row)):
        # missing at end
        # missing indices are (12 - k) .. 11
        if np.all(np.arange(12 - np.sum(row), 12) == np.where(row)[0]):
            end_only[i] = True
    # else could be middle gap
    if not (start_only[i] or end_only[i]):
        middle_gap[i] = True

print('Missing ANY data pattern:')
print(f'  Missing at start only: {start_only.sum()}')
print(f'  Missing at end only:   {end_only.sum()}')
print(f'  Missing in middle gaps:{middle_gap.sum()}')
print()

# Check for gaps in missing months considering circular continuity (Jan-Dec wrap) for ANY data
def has_circular_gap(arr):
    '''Return True if missing months are NOT all contiguous on a circle of 12 months.'''
    idx = np.where(~arr)[0]  # Where condition is False (missing)
    if len(idx) <= 1:
        return False  # 0 or 1 missing month -> no gap
    idx_sorted = np.sort(idx)
    diffs = np.diff(idx_sorted)
    circular_diff = (idx_sorted[0] + 12) - idx_sorted[-1]
    gaps = np.concatenate([diffs, [circular_diff]])
    num_gaps = np.sum(gaps > 1)
    return num_gaps > 1  # more than one gap means missing months are split into multiple arcs

circular_gap_any = np.apply_along_axis(has_circular_gap, 1, (all_present | vv_vh_present_others_missing))
circular_gap_complete = np.apply_along_axis(has_circular_gap, 1, all_present)
circular_gap_vv_vh = np.apply_along_axis(has_circular_gap, 1, vv_vh_present_others_missing)
circular_gap_none = np.apply_along_axis(has_circular_gap, 1, none_present)

print('Proportion of rows with missing months separated by gaps (considering Jan-Dec wrap):')
print(f'ANY data missing:     {circular_gap_any.mean():.3f}')
print(f'COMPLETE data missing:{circular_gap_complete.mean():.3f}')
print(f'ONLY VV/VH missing:   {circular_gap_vv_vh.mean():.3f}')
print(f'NO data missing:      {circular_gap_none.mean():.3f}')

In [ ]:
# -----------------------------------------------------------
# Month availability analysis (ANY data present - different approach)
# -----------------------------------------------------------

# Month has data if ANY of the 12 features is present
has_any_data = np.any(data != -9999, axis=2)      # (n_samples, 12)

print("Shape:", has_any_data.shape)
print("dtype:", has_any_data.dtype)

# Convert bool -> int for arithmetic
A = has_any_data.astype(np.int32)

n_samples = A.shape[0]

# -----------------------------------------------------------
# Marginal probability P(i)
# -----------------------------------------------------------

P = A.mean(axis=0)

print("\nProbability each month has data:")
for m, p in enumerate(P, start=1):
    print(f"Month {m:2d}: {p:.3f}")

# -----------------------------------------------------------
# Joint counts
# -----------------------------------------------------------

joint_counts = A.T @ A

print("\nJoint counts:")
print(joint_counts)

# -----------------------------------------------------------
# Joint probability P(i AND j)
# -----------------------------------------------------------

joint_prob = joint_counts / n_samples

print("\nJoint probability:")
print(np.round(joint_prob, 3))

# -----------------------------------------------------------
# Verify diagonal
# -----------------------------------------------------------

print("\nDiagonal of joint probability:")
print(np.round(np.diag(joint_prob), 3))

print("\nMarginal probability:")
print(np.round(P, 3))

print("\nCheck diagonal equals marginal:")
print(np.allclose(np.diag(joint_prob), P))

# -----------------------------------------------------------
# Conditional probability P(j | i)
# -----------------------------------------------------------

conditional = joint_counts / joint_counts.diagonal()[:, None]

print("\nConditional probability P(j | i):")
print(np.round(conditional, 3))

# -----------------------------------------------------------
# Plot joint probability
# -----------------------------------------------------------

plt.figure(figsize=(8,6))

sns.heatmap(
    joint_prob,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    square=True,
    xticklabels=range(1,13),
    yticklabels=range(1,13),
    vmin=0,
    vmax=joint_prob.max()
)

plt.xlabel("Month j")
plt.ylabel("Month i")
plt.title("Joint Probability P(month i AND month j)")
plt.tight_layout()
plt.show()

# -----------------------------------------------------------
# Plot conditional probability
# -----------------------------------------------------------

plt.figure(figsize=(8,6))

sns.heatmap(
    conditional,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    square=True,
    xticklabels=range(1,13),
    yticklabels=range(1,13),
    vmin=0,
    vmax=1
)

plt.xlabel("Month j")
plt.ylabel("Month i")
plt.title("Conditional Probability P(month j | month i)")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# Month availability (ANY data present - different approach)
# ---------------------------------------------------------

has_any_data = np.any(data != -9999, axis=2)

linear_gaps = []
circular_gaps = []

for row in has_any_data:
    
    months = np.where(row)[0] + 1      # months 1-12
    
    if len(months) < 2:
        continue
    
    # Linear gaps
    linear_gaps.extend(np.diff(months))
    
    # Circular gap (December -> January)
    circular_gap = 12 - months[-1] + months[0]
    circular_gaps.append(circular_gap)

# ---------------------------------------------------------
# Plot linear gaps
# ---------------------------------------------------------

plt.figure(figsize=(8,5))

bins = np.arange(0.5, 12.5, 1)

plt.hist(linear_gaps,
         bins=bins,
         edgecolor='black')

plt.xticks(range(1,12))
plt.xlabel("Gap between consecutive observed months")
plt.ylabel("Frequency")
plt.title("Linear Month Gaps")
plt.grid(alpha=0.3)

plt.show()

# ---------------------------------------------------------
# Plot circular gaps
# ---------------------------------------------------------

plt.figure(figsize=(8,5))

plt.hist(circular_gaps,
         bins=bins,
         edgecolor='black')

plt.xticks(range(1,12))
plt.xlabel("Gap from last observation back to first")
plt.ylabel("Frequency")
plt.title("Circular Month Gap")
plt.grid(alpha=0.3)

plt.show()

# ---------------------------------------------------------
# Summary statistics
# ---------------------------------------------------------

print(f"Mean linear gap    : {np.mean(linear_gaps):.2f}")
print(f"Median linear gap  : {np.median(linear_gaps):.2f}")
print()
print(f"Mean circular gap  : {np.mean(circular_gaps):.2f}")
print(f"Median circular gap: {np.median(circular_gaps):.2f}")

In [ ]:
# -------------------------------------------------------------
# Observed monthly probability of having ANY data
# -------------------------------------------------------------

obs = np.array([
    0.125, 0.252, 0.379, 0.504,
    0.593, 0.641, 0.641, 0.594,
    0.507, 0.379, 0.254, 0.127
])

# -------------------------------------------------------------
# Window lengths
# -------------------------------------------------------------

lengths = [4,5,6]
P_length = np.array([1/3,1/3,1/3])

# -------------------------------------------------------------
# Build matrix A
# Each column corresponds to a possible starting month
# -------------------------------------------------------------

n_months = 12
n_start = 9       # months 1..9

A = np.zeros((n_months, n_start))

for start in range(n_start):
    for L, weight in zip(lengths, P_length):
        if start + L <= 12:
            A[start:start+L, start] += weight

# -------------------------------------------------------------
# Objective function
# -------------------------------------------------------------

def objective(x):
    pred = A @ x
    return np.sum((pred - obs)**2)

# -------------------------------------------------------------
# Constraints
# -------------------------------------------------------------

constraints = ({
    'type':'eq',
    'fun':lambda x: np.sum(x)-1
})

bounds = [(0,None)]*n_start

x0 = np.ones(n_start)/n_start

result = minimize(
    objective,
    x0,
    bounds=bounds,
    constraints=constraints
)

start_prob = result.x
print("Shape of start_prob:", start_prob.shape)

print("Recovered starting month probabilities")

for i,p in enumerate(start_prob,1):
    print(f"Month {i}: {p:.4f}")

print("Predicted monthly availability")
pred_availability = A @ start_prob
print(np.round(pred_availability,3))

print("Observed")
print(np.round(obs,3))

# -------------------------------------------------------------
# Visualization: Plot start_prob, predicted vs observed
# -------------------------------------------------------------

plt.figure(figsize=(14, 5))

# Plot 1: Starting probabilities
plt.subplot(1, 3, 1)
plt.bar(range(1, n_start+1), start_prob, color='steelblue', alpha=0.8)
plt.xlabel('Starting Month')
plt.ylabel('Probability')
plt.title('Recovered Starting Month Probabilities')
plt.xticks(range(1, n_start+1))
plt.grid(alpha=0.3)

# Plot 2: Predicted vs Observed monthly availability
plt.subplot(1, 3, 2)
months = range(1, n_months+1)
plt.plot(months, obs, 'o-', label='Observed', linewidth=2, markersize=8)
plt.plot(months, pred_availability, 's--', label='Predicted', linewidth=2, markersize=8)
plt.xlabel('Month')
plt.ylabel('Probability of Any Data')
plt.title('Observed vs Predicted Monthly Availability')
plt.legend()
plt.grid(alpha=0.3)
plt.ylim(0, 0.7)

# Plot 3: Residuals (Observed - Predicted)
plt.subplot(1, 3, 3)
residuals = obs - pred_availability
plt.bar(months, residuals, color=['red' if r < 0 else 'green' for r in residuals], alpha=0.7)
plt.xlabel('Month')
plt.ylabel('Residual (Obs - Pred)')
plt.title('Prediction Residuals')
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Print RMSE
rmse = np.sqrt(np.mean((obs - pred_availability)**2))
print(f"\nRMSE: {rmse:.6f}")


## Now let's try to simulate the missing data patterns based on the recovered starting month probabilities and the window lengths. We will generate synthetic data to see if we can reproduce the observed monthly availability.

In [ ]:
rng = np.random.default_rng(42)

N = 100000

windows = []

test_start_prob = np.array([0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.00])

for _ in range(N):

    L = rng.choice([4,5,6])

    # Only allow valid start months
    valid = np.arange(1,13-L+1)

    probs = start_prob[:len(valid)]
    #probs = test_start_prob[:len(valid)]
    probs = probs/probs.sum()

    S = rng.choice(valid,p=probs)

    months = np.arange(S,S+L)

    windows.append(months)

# Plot the distributions of windows
#plt.figure(figsize=(10, 6))
plt.hist(np.concatenate(windows), bins=np.arange(1, 14)-0.5, density=True, alpha=0.7, color='steelblue', edgecolor='black')
plt.xticks(range(1, 13))
plt.xlabel('Month')
plt.ylabel('Density')
plt.title('Distribution of Simulated Windows of Data Availability')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Simulation of new model: 4, 5, or 6 months of contiguous data
# Each length equally likely (probability 1/3)
# First month uniformly distributed based on length (no wrapping)

rng = np.random.default_rng(42)

N = 100000

all_months = []  # To collect all months across all simulations
sequence_info = []  # To store (length, start_month) for each sequence

for _ in range(N):
    # Step 1: Choose sequence length (4, 5, or 6) with equal probability
    L = rng.choice([4, 5, 6])
    
    # Step 2: Determine valid start months (1 to 13-L, inclusive)
    valid_starts = np.arange(1, 13 - L + 1)
    
    # Step 3: Choose start month uniformly from valid range
    start_month = rng.choice(valid_starts)
    
    # Step 4: Generate the sequence of months
    months_sequence = np.arange(start_month, start_month + L)
    
    # Store results
    all_months.extend(months_sequence)
    sequence_info.append((L, start_month))

# Convert to numpy arrays for easier handling
all_months = np.array(all_months)
sequence_info = np.array(sequence_info)

print(f"Simulated {N} sequences")
print(f"Total month observations: {len(all_months)}")
print(f"Average months per sequence: {len(all_months)/N:.2f}")

# Plot the distribution of months (similar to previous cells)
plt.figure(figsize=(10, 6))
plt.hist(all_months, bins=np.arange(1, 14)-0.5, density=True, alpha=0.7, color='steelblue', edgecolor='black')
plt.xticks(range(1, 13))
plt.xlabel('Month')
plt.ylabel('Density')
plt.title('Distribution of Months (New Model: 4-6 contiguous months, equal probability)')
plt.grid(alpha=0.3)
plt.show()

# Also show the distribution of sequence lengths
plt.figure(figsize=(8, 5))
lengths = sequence_info[:, 0]
plt.hist(lengths, bins=np.arange(3.5, 7.5, 1), alpha=0.7, color='lightcoral', edgecolor='black', align='mid')
plt.xticks([4, 5, 6])
plt.xlabel('Sequence Length (months)')
plt.ylabel('Frequency')
plt.title('Distribution of Sequence Lengths')
plt.grid(alpha=0.3)
plt.show()

# Show distribution of start months for each length
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
length_labels = ['4 months', '5 months', '6 months']
for i, L in enumerate([4, 5, 6]):
    mask = sequence_info[:, 0] == L
    if np.any(mask):
        starts = sequence_info[mask, 1]
        axes[i].hist(starts, bins=np.arange(0.5, 13-L+1.5, 1), alpha=0.7, color='lightgreen', edgecolor='black', align='mid')
        axes[i].set_xticks(range(1, 13-L+1))
        axes[i].set_xlabel('Start Month')
        axes[i].set_ylabel('Frequency')
        axes[i].set_title(f'Start Month Distribution for {L}-month sequences')
        axes[i].grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Plot raw histogram counts scaled by N and overlay obs from cell-9
plt.figure(figsize=(10, 6))
counts, bin_edges = np.histogram(all_months, bins=np.arange(1, 14)-0.5)
scaled = counts / N  # proportion per month
plt.bar(range(1, 13), scaled, alpha=0.7, color='steelblue', edgecolor='black', label='Simulated proportion')
# obs array from earlier analysis (cell-9)
obs = np.array([0.125, 0.252, 0.379, 0.504, 0.593, 0.641, 0.641, 0.594, 0.507, 0.379, 0.254, 0.127])
plt.plot(range(1, 13), obs, 'o-', color='red', linewidth=2, markersize=8, label='Observed')
plt.xlabel('Month')
plt.ylabel('Proportion')
plt.title('Simulated vs Observed Monthly Proportions')
plt.legend()
plt.grid(alpha=0.3)
plt.show()